In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pandas",
    "pyarrow",
    "fastparquet",
    "nbformat"
])

import io
import json
import re
from collections import defaultdict
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.config import Config
from IPython.display import display
from scipy.optimize import least_squares

# =============================================================================
# 1. 参数区
# =============================================================================
MINIO_ENDPOINT = "https://iseadocker.isea.rwth-aachen.de:9000"
CREDENTIALS_FILE = Path("credentials.json")
BUCKET_NAME = "projects"
BASE_PREFIX = "j8005-metabatt/Metabatt/VTC/20_export_pulse/"
QUALITY_CLASSIFICATION_FILE = Path("电芯整体质量分类v6.csv")

SOH_FILENAME_PATTERN = re.compile(
    r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$",
    flags=re.IGNORECASE,
)
BATTERY_FOLDER_PATTERN = re.compile(
    r"^METABatt_Sony_Murata_18650VTC6_(\d+)/",
    flags=re.IGNORECASE,
)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(str(filename).strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))


# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 3.5

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1
NOMINAL_CURRENT_LEVELS_A = np.array([1.5, 3.0])
CC_CURRENT_DEVIATION_RATIO = 0.03  # 10s/20s前相对早期稳定电流的最大允许偏差
CC_STATUS_VALID = "Valid"
CC_STATUS_INSUFFICIENT = "Insufficient duration"
CC_STATUS_DEVIATION = "Current deviation > 3%"
CC_STATUS_NOT_EVALUATED = "Not evaluated"

# R0 / R1 / R2计算设置
R0_TARGET_AFTER_PAUSE_SEC = 0.5   # pause段最后一个测量点之后外推的R0时间
R1_ANCHOR_SEC = 10.0              # 相对于首个非零电流点
R2_ANCHOR_SEC = 20.0              # 相对于首个非零电流点
R0_FIT_POINT_START = 2            # 默认使用有效pulse第2-6点
R0_FIT_POINT_END = 6              # 拟合终点固定为有效pulse第6点
FIRST_TWO_VOLTAGE_EQUAL_ATOL = 1e-9  # 前两点电压视为相同的绝对容差(V)
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC = 5.0  # pause末点到pulse首点严格大于5s则剔除
R0_EARLY_WINDOW_FLAG = "后段电流不稳定，R0仅使用早期稳定窗口"

# R0置信度规则：
# “干净标签”的有效R0可按权重1.0使用。
TRUSTED_R0_QUALITY_FLAGS = {
    "正常",
    R0_EARLY_WINDOW_FLAG,
    "首点0且电压跳变"
}

R0_CONFIDENCE_FULL = "完全可信（权重1.0）"
R0_CONFIDENCE_REVIEW = "需复核（权重0.5）"
R0_CONFIDENCE_INVALID = "不可用（权重0.0）"

TIME_DIFF_OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0、R1、R2及质量检验列
PULSE_OUTPUT_COLUMNS = TIME_DIFF_OUTPUT_COLUMNS + [
    "Nominal_Current_A",
    "R0",
    "R1",
    "R2",
    "CC_Valid_10s",
    "CC_Valid_20s",
    "CC_10s_Status",
    "CC_20s_Status",
    "CC_Max_Deviation_10s_pct",
    "CC_Max_Deviation_20s_pct",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence"
]


In [2]:
# =============================================================================
# 2. 读取电芯整体质量分类 + 在线 parquet
# =============================================================================

if not QUALITY_CLASSIFICATION_FILE.is_file():
    raise FileNotFoundError(
        f"找不到 {QUALITY_CLASSIFICATION_FILE.resolve()}。"
    )

quality_classification = pd.read_csv(QUALITY_CLASSIFICATION_FILE)


def expand_battery_id_ranges(text):
    battery_ids = []
    for part in str(text).split(","):
        part = part.strip()
        if not part:
            continue

        normalized = part.replace("–", "-").replace("—", "-")
        if "-" in normalized:
            start_text, end_text = normalized.split("-", 1)
            start_id = int(start_text.strip())
            end_id = int(end_text.strip())
            battery_ids.extend(range(start_id, end_id + 1))
        else:
            battery_ids.append(int(normalized))

    return battery_ids


battery_class_rows = []
for _, row in quality_classification.iterrows():
    # 分类文件中“用途=不用”的剔除电芯不进入电阻提取。
    if str(row["用途"]).strip() == "不用":
        continue

    for battery_id in expand_battery_id_ranges(row["电芯文件夹编号范围"]):
        battery_class_rows.append({
            "Battery_ID": battery_id,
            "筛选分类": row["筛选分类"],
            "用途": row["用途"],
        })

battery_classification = (
    pd.DataFrame(battery_class_rows)
    .drop_duplicates(subset=["Battery_ID"])
    .sort_values("Battery_ID")
    .reset_index(drop=True)
)

BATTERY_IDS = set(battery_classification["Battery_ID"].astype(int))


if not CREDENTIALS_FILE.is_file():
    raise FileNotFoundError(
        f"找不到 {CREDENTIALS_FILE.resolve()}。请把 credentials.json 放在 notebook 工作目录中。"
    )

with CREDENTIALS_FILE.open(encoding="utf-8") as credential_file:
    credentials = json.load(credential_file)

try:
    MINIO_ACCESS_KEY = credentials["accessKey"].strip()
    MINIO_SECRET_KEY = credentials["secretKey"].strip()
except (KeyError, AttributeError) as exc:
    raise ValueError("credentials.json 必须包含 accessKey 和 secretKey 字符串。") from exc

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise ValueError("credentials.json 中的 Access Key 或 Secret Key 为空。")

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    region_name="us-east-1",
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)


def list_parquet_objects_by_battery():
    objects_by_battery = defaultdict(list)

    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix=BASE_PREFIX)

    for page in pages:
        for item in page.get("Contents", []):
            object_key = item["Key"]
            if not object_key.lower().endswith(".parquet"):
                continue

            relative_key = object_key[len(BASE_PREFIX):]
            match = BATTERY_FOLDER_PATTERN.match(relative_key)
            if match is None:
                continue

            battery_id = int(match.group(1))
            if battery_id not in BATTERY_IDS:
                continue

            objects_by_battery[battery_id].append(object_key)

    if not objects_by_battery:
        raise FileNotFoundError(
            "未找到质量分类文件中对应电芯的 parquet 文件，请检查 BASE_PREFIX 和连接地址。"
        )

    return {
        battery_id: sorted(object_keys)
        for battery_id, object_keys in sorted(objects_by_battery.items())
    }


def read_one_battery(battery_id, object_keys):
    df_list = []

    for object_key in object_keys:
        filename = object_key.rsplit("/", 1)[-1]

        try:
            soh = extract_soh_from_filename(filename)
        except ValueError:
            continue

        if soh > 100:
            continue

        response = s3.get_object(Bucket=BUCKET_NAME, Key=object_key)
        body = response["Body"]
        try:
            temp = pd.read_parquet(io.BytesIO(body.read()))
        finally:
            body.close()

        temp["File"] = filename
        temp["SOH"] = soh
        df_list.append(temp)

    if not df_list:
        return pd.DataFrame()

    return pd.concat(df_list, ignore_index=True)


objects_by_battery = list_parquet_objects_by_battery()


In [3]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[TIME_DIFF_OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    # 不再因为pulse后段电流下降而提前删除整段数据。
    # R0是否可计算，改为在下面仅依据实际拟合窗口内的电流稳定性判断。

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "Nominal_Current_A": np.nan,
            "R0": np.nan,
            "R1": np.nan,
            "R2": np.nan,
            "CC_Valid_10s": pd.NA,
            "CC_Valid_20s": pd.NA,
            "CC_10s_Status": CC_STATUS_NOT_EVALUATED,
            "CC_20s_Status": CC_STATUS_NOT_EVALUATED,
            "CC_Max_Deviation_10s_pct": np.nan,
            "CC_Max_Deviation_20s_pct": np.nan,
            "R0_Target_Time": pd.NaT,
            "Pause_to_Pulse_Time_Diff_s": np.nan,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]
        segment_current_abs = group["Current"].abs().dropna()
        if not segment_current_abs.empty:
            measured_peak_current = segment_current_abs.max()
            nearest_nominal_pos = np.argmin(
                np.abs(NOMINAL_CURRENT_LEVELS_A - measured_peak_current)
            )
            r0_result["Nominal_Current_A"] = float(
                NOMINAL_CURRENT_LEVELS_A[nearest_nominal_pos]
            )

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]

        pause_to_pulse_gap_s = (pulse_start_time - pause_time).total_seconds()
        r0_result["Pause_to_Pulse_Time_Diff_s"] = pause_to_pulse_gap_s

        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # 大于5s：标记为无效，不再进行长距离反向外推
        if pause_to_pulse_gap_s > MAX_PAUSE_TO_PULSE_GAP_SEC:
            r0_result["R0_Quality"] = "pause结束点到pulse首点时间差>5s"
            return r0_result

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        # 默认使用有效pulse第2-6点进行线性拟合。
        # 若有效pulse第1点与第2点的电压相同（可能是重复采样），
        # 则跳过前两点，改用第3-6点进行线性外推。
        fit_point_start = R0_FIT_POINT_START

        if effective_start_pos + 1 < len(group):
            first_pulse_voltage = group["Voltage"].iloc[effective_start_pos]
            second_pulse_voltage = group["Voltage"].iloc[effective_start_pos + 1]

            first_two_voltage_equal = (
                pd.notna(first_pulse_voltage)
                and pd.notna(second_pulse_voltage)
                and np.isclose(
                    float(first_pulse_voltage),
                    float(second_pulse_voltage),
                    rtol=0.0,
                    atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL
                )
            )

            if first_two_voltage_equal:
                fit_point_start = 3

        fit_start_pos = effective_start_pos + fit_point_start - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(
            subset=["Time", "Voltage", "Current"]
        )

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        # R0只要求实际参与2-6点（或3-6点）拟合的早期窗口电流稳定。
        fit_current_values = fit_points["Current"]
        fit_current_abs_level = round(fit_current_values.abs().iloc[0], 1)
        fit_current_std = fit_current_values.std()

        if fit_current_abs_level == 1.5:
            fit_current_unstable = fit_current_std > STD_LIMIT_1P5A
        elif fit_current_abs_level == 3.0:
            fit_current_unstable = fit_current_std > STD_LIMIT_3A
        else:
            fit_current_unstable = True

        if fit_current_unstable:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：早期拟合窗口电流不稳定"
            )
            return r0_result

        # 若整段电流不稳定、但早期拟合窗口稳定，仍计算R0并添加专用flag。
        if is_bad_current_segment(group):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                R0_EARLY_WINDOW_FLAG
            )

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        # 以首个非零电流点为t=0。CC检查从已通过R0标准差检验的
        # 早期拟合窗口开始，避免把电流阶跃建立过程误判成进入CV。
        anchor_data = group.iloc[effective_start_pos:].dropna(
            subset=["Time", "Voltage", "Current"]
        )
        relative_time_s = (
            anchor_data["Time"] - group["Time"].iloc[effective_start_pos]
        ) / pd.Timedelta(seconds=1)

        early_current_ref = fit_current_values.median()
        fit_start_relative_s = (
            fit_points["Time"].min()
            - group["Time"].iloc[effective_start_pos]
        ) / pd.Timedelta(seconds=1)

        def evaluate_cc_until(anchor_sec):
            if len(anchor_data) < 2 or relative_time_s.max() < anchor_sec:
                return CC_STATUS_INSUFFICIENT, np.nan

            if (
                fit_start_relative_s >= anchor_sec
                or pd.isna(early_current_ref)
                or abs(early_current_ref) <= ZERO_CURRENT_LIMIT
            ):
                return CC_STATUS_NOT_EVALUATED, np.nan

            window_mask = (
                (relative_time_s >= fit_start_relative_s)
                & (relative_time_s <= anchor_sec)
            )
            window_current = anchor_data.loc[window_mask, "Current"].to_numpy()
            anchor_current = np.interp(
                anchor_sec,
                relative_time_s.to_numpy(),
                anchor_data["Current"].to_numpy()
            )
            current_values = np.append(window_current, anchor_current)
            if len(current_values) < 2:
                return CC_STATUS_NOT_EVALUATED, np.nan

            max_deviation_pct = (
                np.max(np.abs(current_values - early_current_ref))
                / abs(early_current_ref)
                * 100
            )

            if max_deviation_pct <= CC_CURRENT_DEVIATION_RATIO * 100:
                return CC_STATUS_VALID, max_deviation_pct

            return CC_STATUS_DEVIATION, max_deviation_pct

        cc_10s_status, cc_max_deviation_10s_pct = evaluate_cc_until(
            R1_ANCHOR_SEC
        )
        cc_20s_status, cc_max_deviation_20s_pct = evaluate_cc_until(
            R2_ANCHOR_SEC
        )
        cc_valid_10s = cc_10s_status == CC_STATUS_VALID
        cc_valid_20s = cc_20s_status == CC_STATUS_VALID
        r0_result["CC_Valid_10s"] = cc_valid_10s
        r0_result["CC_Valid_20s"] = cc_valid_20s
        r0_result["CC_10s_Status"] = cc_10s_status
        r0_result["CC_20s_Status"] = cc_20s_status
        r0_result["CC_Max_Deviation_10s_pct"] = cc_max_deviation_10s_pct
        r0_result["CC_Max_Deviation_20s_pct"] = cc_max_deviation_20s_pct

        if cc_valid_10s:
            voltage_10s = np.interp(
                R1_ANCHOR_SEC,
                relative_time_s.to_numpy(),
                anchor_data["Voltage"].to_numpy()
            )
            r10 = abs((voltage_10s - pause_voltage) / effective_current)
            r0_result["R1"] = r10 - r0_result["R0"]

        if cc_valid_20s:
            voltage_10s, voltage_20s = np.interp(
                [R1_ANCHOR_SEC, R2_ANCHOR_SEC],
                relative_time_s.to_numpy(),
                anchor_data["Voltage"].to_numpy()
            )
            r10 = abs((voltage_10s - pause_voltage) / effective_current)
            r20 = abs((voltage_20s - pause_voltage) / effective_current)
            r0_result["R2"] = r20 - r10

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 在计算R0前剔除极端工况：10% SOC下的3.0A DCH，
        # 以及90% SOC下的3.0A CHA。
        effective_current_abs = round(
            abs(group["Current"].iloc[effective_start_pos]),
            1
        )
        soc = group["SOC"].iloc[0]
        zustand = group["Zustand"].iloc[0]

        if effective_current_abs == 3.0 and (
            (soc == "10%" and zustand == "DCH")
            or (soc == "90%" and zustand == "CHA")
        ):
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    r0_result_columns = [
        "Nominal_Current_A",
        "R0",
        "R1",
        "R2",
        "CC_Valid_10s",
        "CC_Valid_20s",
        "CC_10s_Status",
        "CC_20s_Status",
        "CC_Max_Deviation_10s_pct",
        "CC_Max_Deviation_20s_pct",
        "R0_Target_Time",
        "Pause_to_Pulse_Time_Diff_s",
        "R0_Quality"
    ]

    for column in r0_result_columns:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx, col=column: r0_results[idx][col]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 生成逐条R0置信度
    # -----------------------------
    def classify_r0_confidence(row):
        r0_value = row["R0"]
        quality_text = str(row["R0_Quality"]).strip()

        # 没有有效R0时，无论带有什么flag，都不能参与R0分析。
        if pd.isna(r0_value) or not np.isfinite(r0_value):
            return R0_CONFIDENCE_INVALID

        flags = {
            flag.strip()
            for flag in quality_text.split("；")
            if flag.strip()
        }

        # 只要一条记录所含flag全部属于已验证的干净标签，就给权重1.0。
        if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
            return R0_CONFIDENCE_FULL

        # 将来若新增了仍可计算R0、但尚未验证的flag，先标记为需复核。
        return R0_CONFIDENCE_REVIEW

    pulse_sequence["R0_Confidence"] = pulse_sequence.apply(
        classify_r0_confidence,
        axis=1
    )

    # -----------------------------
    # 9. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence


In [4]:
# =============================================================================
# 4. R0 计算后的脉冲筛选
# =============================================================================
# 筛选脉冲/清除1.5A下的DCH脉冲
def filter_pulse(df):

    pulse_sequence_filter = df[
        ~((df["Zustand"] == "DCH") & (df["Nominal_Current_A"] == 1.5))
    ].copy()
    return pulse_sequence_filter


In [5]:
# =============================================================================
# 5. 按电芯整体质量分类批量提取 R0 / R1 / R2
# =============================================================================

battery_result_list = []

for battery_id, object_keys in objects_by_battery.items():
    df = read_one_battery(battery_id, object_keys)

    if df.empty:
        continue

    pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)
    filtered_pulse = filter_pulse(pulse_sequence)

    class_info = battery_classification.loc[
        battery_classification["Battery_ID"] == battery_id
    ].iloc[0]

    filtered_pulse.insert(0, "Battery_ID", battery_id)
    filtered_pulse.insert(1, "筛选分类", class_info["筛选分类"])
    filtered_pulse.insert(2, "用途", class_info["用途"])

    battery_result_list.append(filtered_pulse)


if battery_result_list:
    resistance_result = pd.concat(
        battery_result_list,
        ignore_index=True
    )
else:
    resistance_result = pd.DataFrame(
        columns=["Battery_ID", "筛选分类", "用途"] + PULSE_OUTPUT_COLUMNS
    )


# 电阻原单位为 Ohm，同时保留 mOhm 列。
resistance_result["R0_mOhm"] = (
    pd.to_numeric(resistance_result["R0"], errors="coerce") * 1000
)
resistance_result["R1_mOhm"] = (
    pd.to_numeric(resistance_result["R1"], errors="coerce") * 1000
)
resistance_result["R2_mOhm"] = (
    pd.to_numeric(resistance_result["R2"], errors="coerce") * 1000
)

result_columns = [
    "Battery_ID",
    "筛选分类",
    "用途",
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "Nominal_Current_A",
    "Voltage",
    "R0",
    "R0_mOhm",
    "R1",
    "R1_mOhm",
    "R2",
    "R2_mOhm",
    "CC_Valid_10s",
    "CC_Valid_20s",
    "CC_10s_Status",
    "CC_20s_Status",
    "CC_Max_Deviation_10s_pct",
    "CC_Max_Deviation_20s_pct",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence",
    "File"
]

resistance_result = (
    resistance_result[result_columns]
    .sort_values(
        ["Battery_ID", "SOH", "SOC", "Zustand", "Current"],
        ascending=[True, False, True, True, True]
    )
    .reset_index(drop=True)
)

display(resistance_result)


,Battery_ID,筛选分类,用途,SOH,SOC,Time,Zustand,Current,Nominal_Current_A,Voltage,...,CC_Valid_20s,CC_10s_Status,CC_20s_Status,CC_Max_Deviation_10s_pct,CC_Max_Deviation_20s_pct,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence,File
0,4,A完整,train/val/test,97.1,10%,2024-09-09 10:09:41.770000+00:00,CHA,1.499757,1.5,3.319417,...,True,Valid,Valid,0.005953,0.011906,2024-09-09 10:09:41.320000+00:00,0.81,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_004_pulse_BM2_9...
1,4,A完整,train/val/test,97.1,10%,2024-09-09 12:11:28.360000+00:00,CHA,2.999464,3.0,3.365002,...,True,Valid,Valid,0.005953,0.011891,2024-09-09 12:11:27.860000+00:00,0.80,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_004_pulse_BM2_9...
2,4,A完整,train/val/test,97.1,50%,2024-09-09 02:09:22.790000+00:00,CHA,1.499846,1.5,3.791951,...,True,Valid,Valid,0.005953,0.005953,2024-09-09 02:09:22.370000+00:00,0.79,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_004_pulse_BM2_9...
3,4,A完整,train/val/test,97.1,50%,2024-09-09 04:11:09.260000+00:00,CHA,2.999643,3.0,3.820192,...,True,Valid,Valid,0.005945,0.005945,2024-09-09 04:11:08.780000+00:00,0.79,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_004_pulse_BM2_9...
4,4,A完整,train/val/test,97.1,50%,2024-09-09 03:10:05.880000+00:00,DCH,-2.999832,3.0,3.714425,...,True,Valid,Valid,0.000000,0.005945,2024-09-09 03:10:05.400000+00:00,0.79,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_004_pulse_BM2_9...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50949,572,A完整,train/val/test,73.9,10%,2026-05-08 23:22:00.470000+00:00,CHA,1.497517,1.5,3.401293,...,True,Valid,Valid,0.126157,0.126157,2026-05-08 23:22:00+00:00,0.89,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_572_pulse_BM37_...
50950,572,A完整,train/val/test,73.9,10%,2026-05-08 21:20:34.040000+00:00,CHA,1.498417,1.5,3.404516,...,True,Valid,Valid,0.066104,0.072108,2026-05-08 21:20:33.470000+00:00,0.87,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_572_pulse_BM37_...
50951,572,A完整,train/val/test,73.9,50%,2026-05-08 15:49:35.910000+00:00,CHA,1.497066,1.5,3.839668,...,True,Valid,Valid,0.138195,0.138195,2026-05-08 15:49:35.330000+00:00,0.87,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_572_pulse_BM37_...
50952,572,A完整,train/val/test,73.9,50%,2026-05-08 13:48:09.690000+00:00,CHA,1.498688,1.5,3.838778,...,True,Valid,Valid,0.048060,0.048060,2026-05-08 13:48:09.100000+00:00,0.86,正常,完全可信（权重1.0）,METABatt_Sony_Murata_18650VTC6_572_pulse_BM37_...
